# 0.LangChain 설치


- 의존성 라이브러리 설치

In [1]:
%pip install -U google-colab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 17.5 MB/s eta 0:00:00


In [2]:
!pip install -qU openai langchain-openai langchain langchain_community
!pip install -qU tiktoken pypdf chromadb faiss-cpu
!pip install -qU langchain-teddynote

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 930.8/930.8 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

- API 키 등록

In [3]:
import os

api_key = "sk-f_Ae5L9K9tiyYUCTa8_wJuF5c194uBcz2IofaZNBxwT3BlbkFJqTsPggjlugRL-et4nfI_ci3QeYCo3NGiuGF1RPNcEA"
os.environ["OPENAI_API_KEY"] = api_key

### `LC01_LangChain 기초, LLM 연동.ipynb`에서 수행한 건 **<U>Zero-Shot Prompting**</U>

# 1.제로샷 프롬프팅 (Zero-Shot Prompting)
- Zero-Shot : AI 모델이 사전 학습 없이 새로운 작업이나 상황에 대응하는 방식
- Zero-Shot Prompting : **사전 예시 없이**, 단순히 요청만 주어져 모델이 답변을 생성하는 방식

> **특징**  
> - **<font color=red>추가 예시 없이</font>** 문제 설명만으로 작업을 수행함  
> - 모델은 오로지 프롬프트에 포함된 정보만을 기반으로 응답을 생성함  
> - **장점** : `빠르고 간단하게 적용할 수 있음`  
> - **단점** : `복잡한 문제나 구체적인 형식을 요구하는 작업에서는 모델의 응답이 기대에 못 미칠 수 있음`

### Zero-Shot Prompting 개선 방법
- Few-Shot Prompting
- 메모리 기능 사용
- 체인 결합 (여러개의 언어 모델 연결)
- RAG (검색, 증강, 생성)
- TAG (테이블(DB), 증강, 생성)
- Agent 등



# 2.퓨샷 프롬프팅 (Few-Shot Prompting)
- 몇 가지 예시(사례)를 프롬프트에 포함시켜, 모델이 원하는 출력 형식이나 문제 해결 방식을 학습하도록 돕는 방식

> **특징**  
> - **<font color=red>예시를 제공</font>**하여 모델에게 어떤 식으로 답변해야 하는지 안내함  
> - 예시를 통해 맥락을 명확하게 전달하고, 응답의 일관성과 정확도를 향상시킴  
> - **장점** : `모델이 복잡하거나 구체적인 작업을 보다 정확하게 수행할 수 있음`  
> - **단점** : `적절한 예시 선택이 중요하며, 프롬프트가 길어질 수 있음`

<br>

---

<br>

##### 현실적으로 LLM을 전이학습하기에는 어려움 → `퓨샷(+메세지)`만으로도 특정 작업에 한해서는 성능이 꽤나 괜찮아짐

In [4]:
from langchain_openai import ChatOpenAI # ChatModel LLM 클래스
from langchain_core.output_parsers import StrOutputParser # 출력파서

from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate
# PromptTemplate : 단일 프롬프트 (일반 템플릿)를 정의
# FewShotPromptTemplate : 여러 예시를 포함한 프롬프트를 정의

In [5]:
# 간단하게 반대말 출력하기 예시 데이터
examples1 = [
    {"input": "happy", "output": "sad"},
    {"input": "tall", "output": "short"},
    {"input": "sunny", "output": "rainy"},
    {"input": "surprised", "output": "calm"},
    {"input": "dry", "output": "humid"},
    {"input": "hot", "output": "cold"},
    {"input": "satisfied", "output": "dissatisfied"}
]

In [6]:
# LLM 모델 초기화
chat_model = ChatOpenAI(
    model = "gpt-4.1-mini", # 모델명 설정
    temperature = 0.2, # LLM 의 자유도 (0일수록 형식적인 1일수록 자유로움)
    max_tokens = 1000  #  프롬프트 출력 토큰 제약
)

In [7]:
# 1. 템플릿 정의 (형식만 지정)
prompt = PromptTemplate.from_template("input : {input} \n output : {output}")



# 2. 퓨샷 프롬프트 정의
fewshot_prompt = FewShotPromptTemplate(
    examples = examples1, # Few Shot 프롬프트용  예시 데이터
    example_prompt = prompt, # 어떤 형식으로 모델한테 보여줄지 지정(템틀릿 객체) 예시 프롬프트
    prefix = "입력의 반대말을 출력한다!", # 예시 설명 문구 (선택)
    suffix = "input : {input}", # 사용자 입력값을 넣을 자리
    input_variables = ["input"] # 실제 입력값으로 치환될 변수(플레이스홀더) 지정
)

# 3. 체인 구성
chain = fewshot_prompt | chat_model | StrOutputParser()  # 체인은 1st prompt => 2nd model에 넘기기 => 3rd parser로 출력하기

# 4. 체인 실행
print(chain.invoke("foggy"))

input : foggy  
output : clear


## 2.1.복잡한 질문에 대해 단계별로 답변을 구성하는 퓨샷 프롬프팅 해보기

In [8]:
examples2 = [
    {
        "question": "상품명: '에어프라이어', 특징: '간편 조리, 1300W 출력', 가격: '120,000원'일 때 단계별 대본 가이드 예시를 보여주세요.",
        "answer": (
            "1) 인사 및 상품 소개: ...\n"
            "2) 주요 특징 설명: ...\n"
            "3) 사용 시나리오 제시: ...\n"
            "4) 가격 및 혜택 안내: ...\n"
            "5) 콜 투 액션: ..."
        )
    },
    {
        "question": "상품명: '블루투스 헤드폰', 특징: '노이즈 캔슬링, 20시간 배터리', 가격: '80,000원'일 때 단계별 대본 가이드 예시를 보여주세요.",
        "answer": (
            "1) 인사 및 관심 유도: ...\n"
            "2) 기능 강조: ...\n"
            "3) 데모 시연 제안: ...\n"
            "4) 특별 할인 안내: ...\n"
            "5) 구매 유도 문구: ..."
        )
    }
]


# FewShot 프롬프팅 주의사항! 퓨샷의 예시는 토큰으로 인식됨!  비용이 증가될 수는 있음


In [9]:
# 1. 일반 프롬프트 템플릿 정의 "Q:{question} \m A:{answer} \n"
prompt = PromptTemplate.from_template("Q:{question} \n A:{answer} \n")


# 2. FewShot 프롬프트 정의
fewshot_prompt = FewShotPromptTemplate(
    examples = examples2,
    example_prompt= prompt,
    # prefix = "라이브 커머스에 활용할 대화야.",
    suffix = "Q:{question}",
    input_variables = ["question"]
)

# 3. 체인 구성

chain = fewshot_prompt | chat_model | StrOutputParser()

# 4. 체인 구동 (stream & stream_response 활용)
# chain.invoke 대신 stream을 사용
from langchain_teddynote.messages import stream_response # 스트림 객체를 출력하는 도구
chain.stream("상품명: '무선 진공청소기', 특징: '강력 흡입, 30분 연속 사용 가능', 가격: '200,000원'일 때 단계별 대본 가이드 예시를 보여주세요.")

temp = stream_response(chain.stream("상품명: '무선 진공청소기', 특징: '강력 흡입, 30분 연속 사용 가능', 가격: '200,000원'일 때 단계별 대본 가이드 예시를 보여주세요."))

temp

A:  
1) 인사 및 상품 소개:  
안녕하세요! 오늘 소개해드릴 제품은 강력한 흡입력과 편리한 무선 사용이 가능한 '무선 진공청소기'입니다.  

2) 주요 특징 설명:  
이 청소기는 강력한 흡입력으로 집안 구석구석 먼지와 이물질을 깨끗하게 제거하며, 30분 연속 사용이 가능해 넓은 공간도 한 번에 청소할 수 있습니다.  

3) 사용 시나리오 제시:  
거실, 침실, 차량 내부 등 다양한 공간에서 자유롭게 사용할 수 있어 청소 시간을 크게 단축시켜줍니다. 특히 무선이라 선에 걸릴 걱정 없이 편리하게 이동하며 청소할 수 있습니다.  

4) 가격 및 혜택 안내:  
현재 가격은 200,000원이며, 한정 기간 동안 특별 할인과 무상 AS 혜택도 함께 제공하고 있습니다.  

5) 콜 투 액션:  
지금 바로 구매하셔서 집안 청소를 더욱 쉽고 빠르게 경험해보세요! 문의사항이 있으시면 언제든지 연락주세요.

In [10]:
 temp # 한번 꺼내버리면 메모리에서 stream 변수에는 아무것도 안남게 된다.
 # 그래서 callbacks 함수들이 나왔다.

## 2.2.Callbacks - 스트리밍(streaming)

- <font color=red>stream_response()</font> : 스트림 형태로 결과 출력
  - **<U>결괏값을 반환하지 않음**</U>
  - import하여 새롭게 초기화 하지 않는 경우, 2번씩 출력되는 문제 발생 `가끔 출력이 2번씩 나오는 문제가 존재함` (버전 업데이트 이전)
  - 다른 Callback 활용 가능

- 스트리밍 옵션은 <font color=red>질의에 대한 답변을 실시간</font>으로 받을 때 유용
- <font color=red>streaming=True</font>로 설정하고 스트리밍으로 답변을 받기 위한 <font color=red>StreamingStdOutCallbackHandler()</font>을 콜백으로 지정

In [11]:
# stream 함수로 출력을 해주는데 결과값을 반환해주는 도구 invoke와 비슷한 형식이다.
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
# 스트리밍 형태로 답변을 출력하면서도, 겨로가값을 다룰 수 있는 콜백함수(유용한도구)!

In [12]:
# 랭체인 구성요소를 설명해주는 예시
example3 = [

    {"input" : "Agent",           "output" : "프롬프트를 통해 도구를 호출하며 워크플로우를 제어할 수 있는 자동화 컴포넌트"},

    {"input" : "Chain",           "output" : "여러 프롬프트와 모델 호출을 순차적으로 연결해 파이프라인을 구성하는 구조"},

    {"input" : "PromptTemplate",  "output" : "입력 변수를 포함해 모델에 보낼 프롬프트 형식을 정의하는 클래스"}

]


In [13]:
# 1. 일반 프롬프트 템플릿
prompt = PromptTemplate.from_template("개념명 : {input} \n 설명 : {output} \n")

In [14]:
# 2. 퓨샷 템플릿객체(프롬프트) 생성
fewshot_prompt = FewShotPromptTemplate(
    examples = example3,
    example_prompt = prompt,
    prefix = "LangChain의 주요 구성요소를 다음 예시처럼 설명하세요.", # pre전 설명문
    suffix = "개념명 : {input}",
    input_variables = ["input"]
)

In [15]:
# 3.스트리밍 콜백 핸들러 생성
Stream_handler = StreamingStdOutCallbackHandler()

In [16]:
# 특정 함수가 실행되었을때 값이 출력되는것을 handler라고 한다.


# 4. 모델 초기화
chat_model = ChatOpenAI(
    model = "gpt-4.1-mini",
    temperature= 0.2,
    streaming= True, # 스트리밍 모듈을 활성화 시켜줘야한다 => callbacks 설정 해줘야한다.
    callbacks= [Stream_handler] # 콜백함수 지정
)

In [17]:
# 5. 체인 구성
chain = fewshot_prompt | chat_model | StrOutputParser()

In [18]:
# 6. 체인 실행
temp = chain.invoke("Tool") # agent 쓰고 싶으면 tool(Rag) 을 쓸 수 있다. tag: 테이블 참조, web:웹 참조, DB: DB참조, API: API 참조

개념명 : Tool  
설명 : 특정 기능이나 API를 수행하는 모듈로, Agent가 호출해 작업을 실행할 수 있는 외부 도구 또는 기능 단위

In [19]:
temp

'개념명 : Tool  \n설명 : 특정 기능이나 API를 수행하는 모듈로, Agent가 호출해 작업을 실행할 수 있는 외부 도구 또는 기능 단위'

## 2.3.ExampleSelector
- 예시가 많으면 토큰 사용량이 많아지고 따라서 API 사용료가 많이 나옴
- 최종 질문과 유사한 예시 한 두 개만 프롬프트에 포함할 예제를 선택하는 경우에 사용

- 종류
  - <font color=red>SemanticSimilarityExampleSelector</font>: 벡터 임베딩의 코사인 유사도를 이용해 질문과 의미가 가까운 예시를 선택
---
- 상황에 따라 사용(별로임)
  - <font color=red>BaseExampleSelector</font>: 상속 받아 사용자 정의 ExampleSelector를 생성
  - <font color=red>LengthBasedExampleSelector</font>: 지정 길이가 넘어가지 않도록 예시 개수를 조절
  - <font color=red>MaxMarginalRelevanceExampleSelector</font>: Maximal Marginal Relevance (MMR)을 이용해 질문과 가까우면서도 다양한 예시를 선택
  - <font color=red>NGramOverlapExampleSelector</font>: n-gram overlap score를 이용해 질문과 가까운 예시를 선택

- 사용자 입력과의 유사도에 따라 예시를 선택해서 LLM 모델에 Few-show Prompting 하기 (질문에 대해 예시와 같이 답변하도록 하는 기능 구현)
  - 예시와 사용자 입력을 Text Embeddings 변환하여, Cosine Similarity가 가장 높은 k개의 예시를 선별해서 Few-shot Prompt에 넣어서 LLM 모델에 전달하는 방법

In [20]:

from langchain.prompts.example_selector import SemanticSimilarityExampleSelector


# 임베딩 벡터화한것을 저장하는 라이브러리 불러오자
from langchain_openai import OpenAIEmbeddings # 임베딩 클래스 (text to num(vector)로 표현)

# 담아두는 역할
from langchain.vectorstores import FAISS # 벡터 데이터베이스 (벡터화 된 예시들을 담아둘 공간), 유사도 검사시 활용



In [21]:
# 첫번째 예시: 회의록을 전달하면 정리된 회의록을 출력하는 예시

# 두번째 예시: 보고서를 전달하면 핵심요약을 출력하는 예시

examples4 = [

    {

        "input": (

            "2023년 12월 25일, XYZ 회사의 마케팅 전략 회의가 오후 3시에 시작되었다. "

            "회의에는 마케팅 팀장인 김수진, 디지털 마케팅 담당자인 박지민, 소셜 미디어 관리자인 이준호가 참석했다. "

            "회의의 주요 목적은 2024년 상반기 마케팅 전략을 수립하고, 새로운 소셜 미디어 캠페인에 대한 아이디어를 논의하는 것이었다. "

            "팀장인 김수진은 최근 시장 동향에 대한 간략한 개요를 제공했으며, 이어서 각 팀원이 자신의 분야에서의 전략적 아이디어를 발표했다."

        ),

        "answer": (

            "회의록: XYZ 회사 마케팅 전략 회의\n"

            "일시: 2023년 12월 25일\n"

            "장소: XYZ 회사 회의실\n"

            "참석자: 김수진 (마케팅 팀장), 박지민 (디지털 마케팅 담당자), 이준호 (소셜 미디어 관리자)\n\n"

            "1. 개회\n"

            "   - 김수진 팀장의 개회사\n"

            "2. 시장 동향 개요 (김수진)\n"

            "   - 최근 시장 동향 분석\n"

            "3. 디지털 마케팅 전략 (박지민)\n"

            "   - SEO 최적화 및 온라인 광고 방안\n"

            "4. 소셜 미디어 캠페인 (이준호)\n"

            "   - 인플루언서 마케팅 제안\n"

            "5. 종합 논의\n"

            "   - 예산 및 자원 배분\n"

            "6. 마무리\n"

            "   - 다음 회의 일정 및 회의록 배포 담당자 지정"

        )

    },

    {

        "input": (

            "이 문서는 '지속 가능한 도시 개발을 위한 전략'에 대한 20페이지 분량의 보고서입니다. "

            "보고서는 지속 가능한 도시 개발의 중요성, 현재 도시화의 문제점, 그리고 지속 가능한 개발 전략을 다루고 있습니다. "

            "여러 국가의 성공 사례와 얻은 교훈도 포함되어 있습니다."

        ),

        "answer": (

            "문서 요약: 지속 가능한 도시 개발 전략 보고서\n\n"

            "- 중요성: 사회적·경제적·환경적 이점 강조\n"

            "- 문제점: 환경 오염, 자원 고갈, 불평등 분석\n"

            "- 전략: 친환경 건축, 대중교통 개선, 에너지 효율성 증대\n"

            "- 사례 연구: 코펜하겐, 요코하마 성공 사례\n"

            "- 교훈: 다각적 접근, 지역사회 협력, 장기 계획 필요"

        )

    }

]


In [22]:
# 예시 선택기 생성
example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples4,          # 예시
    OpenAIEmbeddings(), # 임베딩 모델 (예시를 벡터로 변환)
    FAISS,              # 유사도 계산을 수행하기 위해 활용하는 저장소
    k = 1               # 입력과 가장 가까운 예시 1개만 선택
)




In [23]:
# 일반 템플릿 생성
prompt = PromptTemplate.from_template(
    """
    문서: {input}

    결과: {answer}
    """
)

In [24]:
# 퓨샷 템플릿 생성
fewshot_prompt = FewShotPromptTemplate(
    example_selector = example_selector, # 예시를 직접적으로 전달하는게 아닌, 예시 선택기 사용하여 전달!
    example_prompt = prompt, # 예시 프롬프트
    prefix = "아래 예시를 참고해서, 입력된 텍스트에 대해 적절한 형식으로 회의록 또는 요약을 생성해",
    suffix = "문서: {input}",
    input_variables = ["input"]
)

In [25]:
# 모델은 위에서 정의한 chat_model(스트리밍 콜백 지정) 활용


In [26]:
# 체인 구성
chain = fewshot_prompt | chat_model | StrOutputParser()

In [27]:
# 첫번째 회의록
proceedings = {

    "input": (

        "2025년 5월 20일 오후 4시부터 약 90분간, 서울 본사 3층 대회의실에서 "

        "신제품 출시 킥오프 회의가 진행되었습니다. 회의에는 개발팀장 김민준, "

        "디자인팀장 이지은, 영업팀장 박성호, 마케팅팀장 오세훈, 품질관리 담당 최유리가 참석했으며, "

        "주요 안건은 제품 디자인 확정, 출시 일정, 마케팅 채널 전략, 리스크 관리 방안이었습니다."

    )

}
chain_proceedings = chain.invoke(proceedings)



회의록: 신제품 출시 킥오프 회의  
일시: 2025년 5월 20일 오후 4시 ~ 약 90분  
장소: 서울 본사 3층 대회의실  
참석자: 김민준 (개발팀장), 이지은 (디자인팀장), 박성호 (영업팀장), 오세훈 (마케팅팀장), 최유리 (품질관리 담당)  

1. 개회  
   - 회의 시작 및 참석자 소개  

2. 제품 디자인 확정 (이지은)  
   - 최종 디자인 시안 검토 및 확정  
   - 디자인 관련 추가 수정 사항 논의  

3. 출시 일정 (김민준)  
   - 제품 개발 진행 상황 보고  
   - 출시 목표 일정 공유 및 조율  

4. 마케팅 채널 전략 (오세훈)  
   - 주요 마케팅 채널 선정 및 활용 방안  
   - 프로모션 계획 및 타겟 고객층 설정  

5. 리스크 관리 방안 (최유리)  
   - 품질 관리 및 문제 발생 시 대응 계획  
   - 리스크 사전 예방 대책 논의  

6. 종합 논의  
   - 각 부서 간 협력 방안 및 일정 조율  
   - 추가 지원 필요 사항 확인  

7. 마무리  
   - 다음 회의 일정 조율  
   - 회의록 작성 및 배포 담당자 지정

In [28]:
# 두번째 보고서
report={

    "input": (

        "이 문서는 '머신러닝 모델 성능 최적화 기법'에 대한 15페이지 분량의 기술 보고서입니다. "

        "보고서는 하이퍼파라미터 튜닝(그리드 서치, 베이지안 최적화), 데이터 증강(회전·크롭·색상 보정), "

        "앙상블 기법(배깅·부스팅·스태킹), 모델 경량화(지식 증류·양자화), 성능 평가(Accuracy·Precision·Recall·F1) 등을 상세히 다룹니다."

    )
}

chain_report = chain.invoke(report)

문서 요약: 머신러닝 모델 성능 최적화 기법 기술 보고서

- 하이퍼파라미터 튜닝: 그리드 서치, 베이지안 최적화 방법 소개  
- 데이터 증강: 회전, 크롭, 색상 보정 기법 설명  
- 앙상블 기법: 배깅, 부스팅, 스태킹 활용 방안  
- 모델 경량화: 지식 증류, 양자화 기법 적용  
- 성능 평가: Accuracy, Precision, Recall, F1 지표 분석

## 2.4.MessagePromptTemplate 활용 - 역할을 부여하는 것
- <font color=red>AI 메시지, 시스템 메시지, 사용자 메시지를 사용</font>하여 적절한 맥락을 유지하고, 사용자와의 상호작용을 보다 자연스럽게 처리
  - <font color=red>SystemMessagePromptTemplate</font> : 시스템 프롬프트 설정
  - <font color=red>HumanMessagePromptTemplate</font> : 사용자 프롬프트 설정
- <font color=red>ChatPromptTemplate.from_messages()</font> : 시스템 메시지와 사용자 메시지 템플릿을 포함하는 챗 프롬프트를 구성
- <font color=red>chat_prompt.format_messages()</font> : 사용자의 질문을 포함한 메시지 리스트를 동적으로 생성하여 반환

- 생성된 메시지 리스트는 대화형 인터페이스나 언어 모델과의 상호작용을 위한 입력으로 사용
- 각 메시지는 <font color=red>role</font> (메시지를 말하는 주체, 여기서는 system 또는 user)과 <font color=red>content</font> (메시지의 내용) 속성을 포함
- 이 구조는 시스템과 사용자 간의 대화 흐름을 명확하게 표현하며, 언어 모델이 이를 기반으로 적절한 응답을 생성할 수 있도록 지원

In [29]:
from langchain_core.prompts import ChatPromptTemplate # 메세지 프롬프팅 전용 템플릿

In [30]:
# 1. 메세지 리스트 정의
message = [("system", "너는 천문학 전문가이고, 천문학 관련 질문에 전문가 수준으로 대답한다."),
           ("human", "{user_input}")
]
# 2. 메세지 템플릿 객체 생성
chat_prompt = ChatPromptTemplate.from_messages(message)
# 3. 체인 구성
chain = chat_prompt | chat_model | StrOutputParser()
# 4. 체인 구동

chain.invoke({"user_input" : "지구는 어떤 행성이야?"})

지구는 태양계에서 세 번째로 태양에 가까운 행성이며, 태양으로부터 평균 약 1억 4,960만 킬로미터(1 AU) 떨어져 있습니다. 지구는 암석질 행성(내행성)으로, 주로 규산염 암석과 금속으로 이루어져 있습니다. 지구의 주요 특징은 다음과 같습니다:

1. **크기와 구조**: 지구의 지름은 약 12,742km로, 태양계 내에서 다섯 번째로 큰 행성입니다. 내부는 중심핵(주로 철과 니켈), 맨틀, 그리고 지각으로 구성되어 있습니다.

2. **대기**: 지구는 질소(약 78%)와 산소(약 21%)를 주성분으로 하는 대기를 가지고 있으며, 이는 생명체가 존재하는 데 필수적인 역할을 합니다.

3. **물의 존재**: 지구 표면의 약 71%가 물로 덮여 있으며, 액체 상태의 물이 풍부하게 존재하는 유일한 태양계 행성입니다. 이는 생명체가 번성할 수 있는 중요한 조건입니다.

4. **생명체**: 현재까지 알려진 바로는 지구만이 생명체가 존재하는 행성입니다. 다양한 생태계와 복잡한 생명체가 공존하고 있습니다.

5. **자전과 공전**: 지구는 약 24시간에 한 바퀴 자전하며, 이로 인해 낮과 밤이 생깁니다. 태양 주위를 약 365.25일에 한 바퀴 공전하여 계절 변화를 일으킵니다.

6. **자기장**: 지구는 강한 자기장을 가지고 있어 태양풍으로부터 대기를 보호하고, 나침반이 작동하는 기반이 됩니다.

이러한 특성들 덕분에 지구는 현재까지 발견된 행성 중에서 생명체가 존재하기에 가장 적합한 환경을 제공하고 있습니다.

'지구는 태양계에서 세 번째로 태양에 가까운 행성이며, 태양으로부터 평균 약 1억 4,960만 킬로미터(1 AU) 떨어져 있습니다. 지구는 암석질 행성(내행성)으로, 주로 규산염 암석과 금속으로 이루어져 있습니다. 지구의 주요 특징은 다음과 같습니다:\n\n1. **크기와 구조**: 지구의 지름은 약 12,742km로, 태양계 내에서 다섯 번째로 큰 행성입니다. 내부는 중심핵(주로 철과 니켈), 맨틀, 그리고 지각으로 구성되어 있습니다.\n\n2. **대기**: 지구는 질소(약 78%)와 산소(약 21%)를 주성분으로 하는 대기를 가지고 있으며, 이는 생명체가 존재하는 데 필수적인 역할을 합니다.\n\n3. **물의 존재**: 지구 표면의 약 71%가 물로 덮여 있으며, 액체 상태의 물이 풍부하게 존재하는 유일한 태양계 행성입니다. 이는 생명체가 번성할 수 있는 중요한 조건입니다.\n\n4. **생명체**: 현재까지 알려진 바로는 지구만이 생명체가 존재하는 행성입니다. 다양한 생태계와 복잡한 생명체가 공존하고 있습니다.\n\n5. **자전과 공전**: 지구는 약 24시간에 한 바퀴 자전하며, 이로 인해 낮과 밤이 생깁니다. 태양 주위를 약 365.25일에 한 바퀴 공전하여 계절 변화를 일으킵니다.\n\n6. **자기장**: 지구는 강한 자기장을 가지고 있어 태양풍으로부터 대기를 보호하고, 나침반이 작동하는 기반이 됩니다.\n\n이러한 특성들 덕분에 지구는 현재까지 발견된 행성 중에서 생명체가 존재하기에 가장 적합한 환경을 제공하고 있습니다.'

## 2.5.FewShotChatMessagePromptTemplate - 최종 완성본
- <font color=red>Example Selector를 설정해주고, ChatPromptTemplate.from_messages()에 "human"과 "ai"의 메시지 포맷을 설정</font>
- 입력으로 받는 변수 input_variables=["input"] 을 설정

- Example Selector 의 유사도 검색 문제 해결
  - 유사도 계산시 instruction 과 input 을 사용하고 있지만, instruction 만 사용하여 검색시 제대로된 유사도 결과가 나오지 않음
  - 이를 해결하기 위해 커스텀 유사도 계산을 위한 클래스를 정의하여 사용

In [31]:
# 크로마 벡터디비 사용하겠다.
from langchain.vectorstores import Chroma # 벡터 DB
# FAISS보다 커스터마이즈 해준 적당한 chroma가 적당하데


In [32]:
# 첫번째 회의록, 두번째 보고서 예시# prefix에는 instruction 넣는데
examples5 = [

    {

        "instruction": "당신은 회의록 작성 전문가 입니다. 주어진 정보를 바탕으로 회의록을 작성해 주세요",

        "input": "2023년 12월 25일, XYZ 회사의 마케팅 전략 회의가 오후 3시에 시작되었다. 회의에는 마케팅 팀장인 김수진, 디지털 마케팅 담당자인 박지민, 소셜 미디어 관리자인 이준호가 참석했다. 회의의 주요 목적은 2024년 상반기 마케팅 전략을 수립하고, 새로운 소셜 미디어 캠페인에 대한 아이디어를 논의하는 것이었다. 팀장인 김수진은 최근 시장 동향에 대한 간략한 개요를 제공했으며, 이어서 각 팀원이 자신의 분야에서의 전략적 아이디어를 발표했다.",

        "answer": """

회의록: XYZ 회사 마케팅 전략 회의

일시: 2023년 12월 25일

장소: XYZ 회사 회의실

참석자: 김수진 (마케팅 팀장), 박지민 (디지털 마케팅 담당자), 이준호 (소셜 미디어 관리자)

1. 개회

   - 회의는 김수진 팀장의 개회사로 시작됨.

   - 회의의 목적은 2024년 상반기 마케팅 전략 수립 및 새로운 소셜 미디어 캠페인 아이디어 논의.

2. 시장 동향 개요 (김수진)

   - 김수진 팀장은 최근 시장 동향에 대한 분석을 제시.

   - 소비자 행동 변화와 경쟁사 전략에 대한 통찰 공유.

3. 디지털 마케팅 전략 (박지민)

   - 박지민은 디지털 마케팅 전략에 대해 발표.

   - 온라인 광고와 SEO 최적화 방안에 중점을 둠.

4. 소셜 미디어 캠페인 (이준호)

   - 이준호는 새로운 소셜 미디어 캠페인에 대한 아이디어를 제안.

   - 인플루언서 마케팅과 콘텐츠 전략에 대한 계획을 설명함.

5. 종합 논의

   - 팀원들 간의 아이디어 공유 및 토론.

   - 각 전략에 대한 예산 및 자원 배분에 대해 논의.

6. 마무리

   - 다음 회의 날짜 및 시간 확정.

   - 회의록 정리 및 배포는 박지민 담당.

"""

    },

    {

        "instruction": "당신은 요약 전문가 입니다. 다음 주어진 정보를 바탕으로 내용을 요약해 주세요",

        "input": "이 문서는 '지속 가능한 도시 개발을 위한 전략'에 대한 20페이지 분량의 보고서입니다. 보고서는 지속 가능한 도시 개발의 중요성, 현재 도시화의 문제점, 그리고 도시 개발을 지속 가능하게 만들기 위한 다양한 전략을 포괄적으로 다루고 있습니다. 이 보고서는 또한 성공적인 지속 가능한 도시 개발 사례를 여러 국가에서 소개하고, 이러한 사례들을 통해 얻은 교훈을 요약하고 있습니다.",

        "answer": """

문서 요약: 지속 가능한 도시 개발을 위한 전략 보고서

- 중요성: 지속 가능한 도시 개발이 필수적인 이유와 그에 따른 사회적, 경제적, 환경적 이익을 강조.

- 현 문제점: 현재의 도시화 과정에서 발생하는 주요 문제점들, 예를 들어 환경 오염, 자원 고갈, 불평등 증가 등을 분석.

- 전략: 지속 가능한 도시 개발을 달성하기 위한 다양한 전략 제시. 이에는 친환경 건축, 대중교통 개선, 에너지 효율성 증대, 지역사회 참여 강화 등이 포함됨.

- 사례 연구: 전 세계 여러 도시의 성공적인 지속 가능한 개발 사례를 소개. 예를 들어, 덴마크의 코펜하겐, 일본의 요코하마 등의 사례를 통해 실현 가능한 전략들을 설명.

- 교훈: 이러한 사례들에서 얻은 주요 교훈을 요약. 강조된 교훈에는 다각적 접근의 중요성, 지역사회와의 협력, 장기적 계획의 필요성 등이 포함됨.

이 보고서는 지속 가능한 도시 개발이 어떻게 현실적이고 효과적인 형태로 이루어질 수 있는지에 대한 심도 있는 분석을 제공합니다.

"""

    }

]


In [33]:
# 2025.09.04 확인 => chroma DB import가 아래와 같이 변결될 예정
!pip -q install -U langchain-chroma

In [34]:
# 퓨샷 + 메세지 프롬프팅 전용 클래스 import
from langchain_core.prompts import FewShotChatMessagePromptTemplate
from langchain.vectorstores import Chroma

In [35]:
from langchain_chroma import Chroma

In [36]:
# 1. 벡터 DB 초기화
# FAISS 는 뭔가 안맞다네?
chroma = Chroma(
    "fewshot_chat",
    OpenAIEmbeddings()
    # persist_directory="./chroma_db"
)


In [37]:
# 2. 메세지 템플릿 생성
messages = [
    ("system", "{instruction}"),
    ("human", "{input}"), # input prompt
    ("ai", "{answer}")  # output prompt 답변 만들어낸 예시
]

example_prompt = ChatPromptTemplate.from_messages(messages)
example_prompt

ChatPromptTemplate(input_variables=['answer', 'input', 'instruction'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['instruction'], input_types={}, partial_variables={}, template='{instruction}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}), AIMessagePromptTemplate(prompt=PromptTemplate(input_variables=['answer'], input_types={}, partial_variables={}, template='{answer}'), additional_kwargs={})])

In [38]:
# 3. 예시 선택기 생성
example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples5,
    OpenAIEmbeddings(),
    chroma,
    k=1
)

In [39]:
# 4. 퓨샷 템플릿 생성 (2가지 인자만 넣어주면 된다네~)
fewshot_message_prompt = FewShotChatMessagePromptTemplate(
    example_selector = example_selector,
    example_prompt = example_prompt

)


In [40]:
# 실제 질의(사용자 입력값)
question = {

    "instruction": "회의록을 작성해 주세요",

    "input": (

        "2023년 12월 26일, ABC 기술 회사의 제품 개발 팀은 새로운 모바일 애플리케이션 프로젝트에 대한 "

        "주간 진행 상황 회의를 가졌다. 이 회의에는 프로젝트 매니저인 최현수, 주요 개발자인 황지연, "

        "UI/UX 디자이너인 김태영이 참석했다. 회의의 주요 목적은 프로젝트의 현재 진행 상황을 검토하고, "

        "다가오는 마일스톤에 대한 계획을 수립하는 것이었다. 각 팀원은 자신의 작업 영역에 대한 업데이트를 제공했고, "

        "팀은 다음 주까지의 목표를 설정했다."

    ),

}


In [41]:
# 모델 - 위에서 정의해둔 chat_model 활용하겠다

In [42]:
# 5. 최종 프롬프트 조립
## 순서 : 시스템 지시 => 유사예시 => 사용자 입력
final_prompt = ChatPromptTemplate.from_messages(
    [("system", "{instruction}"),
     fewshot_message_prompt, # Use the fewshot_message_prompt directly here
     ("human", "{input}")]
)

In [43]:
# 6. 체인구성
chain = final_prompt | chat_model | StrOutputParser()

In [44]:
chain.invoke(question) # 회의록에 알맞다라고 판단해서 회의록형태로 작성해준것으로 output이 된다.

회의록: ABC 기술 회사 제품 개발 팀 주간 진행 상황 회의

일시: 2023년 12월 26일

장소: ABC 기술 회사 회의실

참석자: 최현수 (프로젝트 매니저), 황지연 (주요 개발자), 김태영 (UI/UX 디자이너)

1. 개회

   - 최현수 프로젝트 매니저가 회의를 시작하며 목적을 설명.

   - 회의 목적: 프로젝트 현재 진행 상황 검토 및 다가오는 마일스톤 계획 수립.

2. 진행 상황 업데이트

   - 황지연: 주요 개발 작업 현황 보고, 현재까지 완료된 기능 및 해결된 이슈 공유.

   - 김태영: UI/UX 디자인 진행 상황 발표, 사용자 피드백 반영 및 디자인 수정 내용 설명.

3. 마일스톤 계획

   - 팀은 다가오는 마일스톤에 맞춰 다음 주까지 달성해야 할 목표 설정.

   - 각 팀원의 역할과 책임 재확인.

4. 논의 및 질의응답

   - 프로젝트 일정 조정 및 리소스 배분에 관한 토론.

   - 발생 가능한 위험 요소 및 대응 방안 논의.

5. 마무리

   - 다음 주 진행 상황 보고 일정 확정.

   - 회의 종료 및 후속 조치 안내.

'회의록: ABC 기술 회사 제품 개발 팀 주간 진행 상황 회의\n\n일시: 2023년 12월 26일\n\n장소: ABC 기술 회사 회의실\n\n참석자: 최현수 (프로젝트 매니저), 황지연 (주요 개발자), 김태영 (UI/UX 디자이너)\n\n1. 개회\n\n   - 최현수 프로젝트 매니저가 회의를 시작하며 목적을 설명.\n\n   - 회의 목적: 프로젝트 현재 진행 상황 검토 및 다가오는 마일스톤 계획 수립.\n\n2. 진행 상황 업데이트\n\n   - 황지연: 주요 개발 작업 현황 보고, 현재까지 완료된 기능 및 해결된 이슈 공유.\n\n   - 김태영: UI/UX 디자인 진행 상황 발표, 사용자 피드백 반영 및 디자인 수정 내용 설명.\n\n3. 마일스톤 계획\n\n   - 팀은 다가오는 마일스톤에 맞춰 다음 주까지 달성해야 할 목표 설정.\n\n   - 각 팀원의 역할과 책임 재확인.\n\n4. 논의 및 질의응답\n\n   - 프로젝트 일정 조정 및 리소스 배분에 관한 토론.\n\n   - 발생 가능한 위험 요소 및 대응 방안 논의.\n\n5. 마무리\n\n   - 다음 주 진행 상황 보고 일정 확정.\n\n   - 회의 종료 및 후속 조치 안내.'

# 3.여러 개의 프롬프트 연결하기
- PromptTemplate을 이용해서 <font color=red>+</font> 연산자를 이용해서 여러개의 템플릿을 연결
  - 문자열 + 문자열
  - PromptTemplate + PromptTemplate
  - PromptTemplate + 문자열

---
- 프롬프트 조립용

In [45]:
# 1. 일반 템플릿 정의
prompt_template = PromptTemplate.from_template("{area1}과 {area2}의 시차는 몇이야?")

In [46]:
# 2. 결합
combined_prompt = (prompt_template + PromptTemplate.from_template("\n{transport}로 얼마나 걸릴까?")
                   + "\n{language}로 번역해주세요")

In [47]:
combined_prompt

PromptTemplate(input_variables=['area1', 'area2', 'language', 'transport'], input_types={}, partial_variables={}, template='{area1}과 {area2}의 시차는 몇이야?\n{transport}로 얼마나 걸릴까?\n{language}로 번역해주세요')

In [48]:
# 체인 구성
chain = combined_prompt | chat_model | StrOutputParser()

In [49]:
chain.invoke({"area1": "호주", "area2": "뉴욕", "transport": "비행기", "language": "영어"})

호주와 뉴욕의 시차는 몇 시간인가요?  
비행기로 얼마나 걸리나요?

What is the time difference between Australia and New York?  
How long does it take by plane?

'호주와 뉴욕의 시차는 몇 시간인가요?  \n비행기로 얼마나 걸리나요?\n\nWhat is the time difference between Australia and New York?  \nHow long does it take by plane?'

# 4.순차적인 체인 연결
- runableparellel 은 병렬 수행
- 첫 번째 체인 : 한국어 단어를 영어로 번역하는 작업을 수행
- 두 번째 체인 : 해당 단어를 설명하는 작업을 수행

In [53]:
# 첫 번째 템플릿
prompt1 = PromptTemplate.from_template("{kr_word}를 영어로 번역해줘.")

In [56]:
# 두 번째 템플릿
prompt2 = PromptTemplate.from_template("{eng_word}를 Oxford 사전 기반으로 한글로 설명해줘.")

In [52]:
chat_model

ChatOpenAI(callbacks=[<langchain_core.callbacks.streaming_stdout.StreamingStdOutCallbackHandler object at 0x7d54a736f740>], client=<openai.resources.chat.completions.completions.Completions object at 0x7d54a781c080>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7d54a73a62d0>, root_client=<openai.OpenAI object at 0x7d54a7699eb0>, root_async_client=<openai.AsyncOpenAI object at 0x7d54a73a5580>, model_name='gpt-4.1-mini', temperature=0.2, model_kwargs={}, openai_api_key=SecretStr('**********'), streaming=True)

In [54]:
# 첫 번째 체인   => kr_word : 미래 입력 => prompt1 => chat_model => StrOutputParser 에 번역되어 prompt1
chain1 = prompt1 | chat_model | StrOutputParser()

In [58]:
# 두 번째 체인 => prompt1 => prompt2 => chat_model => StrOutputParser => chain2
chain2 =  {"eng_word":chain1} | prompt2 | chat_model | StrOutputParser()

In [59]:
# 사용해보기
result = chain2.invoke({"kr_word": "사과"})

"사과"는 영어로 "apple"입니다."사과"는 영어 단어 "apple"에 해당합니다. Oxford 사전에 따르면, "apple"은 둥글고 보통 빨간색, 초록색 또는 노란색을 띠는 과일로, 나무에서 자라며 식용으로 널리 소비됩니다. 따라서 "사과"는 이러한 과일을 가리키는 한국어 단어입니다.